# Task: post a generation

Generate a random colour image (themed by the active vote's winning theme, if any) and post it to Bluesky. Headless version of the repo-root `post.py`, run by `core.scheduler`.

In [ ]:
dry_run = False  # build the image but skip the Bluesky post

In [ ]:
import os
import sys

# add the repo root to sys.path so `import config` / `from core...` resolve
# when this notebook is run directly (e.g. from VSCode); when the scheduler
# runs it via papermill the working directory is already the root, so this is
# a harmless no-op.
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, 'pyproject.toml')) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

In [ ]:
import io

import atproto

import config
from core.interactions import InteractionsService
from core.pipeline.pipeline import Pipeline
from core.pipeline.enums import Theme

In [ ]:
# the winning theme while it is active (its theme window), else the default
theme = InteractionsService().vote.active or Theme.DEFAULT

image, colours = (
    Pipeline()
    .filter(theme)
    .random()
    .generate()
)

buffer = io.BytesIO()
image.save(buffer, format="PNG")

print('generated with theme:', theme)
print('colours:', [c.name for c in colours])

In [ ]:
alt_text = (
    f"A picture of the following colors: {[c.name for c in colours]}. "
    f"Their hex codes are {[c.hexcode for c in colours]}."
)

if dry_run:
    print('dry_run: generated image only, not posting')
else:
    client = atproto.Client()
    client.login(config.ATPROTO_CLIENT_USERNAME, config.ATPROTO_CLIENT_PASSWORD)
    client.send_image("", image=buffer.getvalue(), image_alt=alt_text)
    print('posted generation')